# Fraud Detection with CatBoost (optimized)

**What changed vs. the previous version**

| Area | Before | Now |
|---|---|---|
| Features | raw columns + log transforms (no effect on trees) | ratio, cyclical-time and interaction features |
| Tuning | 6 random trials, 3-fold, no early stopping | Optuna (TPE + pruning), early stopping, tunes class-weight strength |
| Training data | 64% of data | 100% of the training split (5-fold ensemble) |
| Threshold | single 20% validation split | out-of-fold predictions on all training rows (more stable) |
| Evaluation | AP + report | AP, ROC-AUC, MCC, recall@precision, PR curve, bootstrap CI, feature importance |

Accuracy is a misleading metric for rare fraud (predicting "never fraud" scores ~99%), so the notebook optimizes **average precision (PR-AUC)** and reports precision / recall / F1 at a validation-chosen threshold.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from catboost import CatBoostClassifier
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    matthews_corrcoef,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ---------------- Configuration ----------------
RANDOM_STATE = 42
DATA_PATH = Path("../Datasets/Transaction_risk_insights_feature_engineered.csv")
N_TRIALS = 30            # more trials = better params (try 50+ if time allows)
SEARCH_ROWS = 150_000    # stratified subsample used while tuning
TUNE_FOLDS = 3
FINAL_FOLDS = 5          # number of models in the final ensemble
MAX_ITERATIONS = 2000    # upper bound; early stopping picks the real number
MIN_PRECISION_TARGET = 0.80   # for the "high-precision" operating point


## 1. Load and inspect the data

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("Missing values per column (top):")
print(df.isna().sum().sort_values(ascending=False).head(5).to_string())
df.head()


## 2. Feature engineering

Trees are invariant to monotonic transforms like `log1p`, so those were dropped. Instead we add features that give the model information it can't easily derive itself: **ratios** (amount vs. balance / EMI), **cyclical time** encodings and a few **categorical interactions**. Everything is row-wise (no target use, no cross-row statistics), so there is no leakage.

In [ ]:
TARGET = "is_fraud"
BASE_FEATURES = [
    "account_type", "transaction_type", "transaction_amount", "transaction_direction",
    "account_balance", "merchant_category", "state", "credit_score", "has_loan",
    "loan_type", "emi_amount", "channel", "kyc_status", "transaction_hour",
    "transaction_month", "day_of_week", "is_weekend",
]
missing_columns = sorted(set(BASE_FEATURES + [TARGET]) - set(df.columns))
if missing_columns:
    raise KeyError(f"Missing required dataset columns: {missing_columns}")


def build_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Row-wise feature engineering (safe to apply before splitting)."""
    out = frame.loc[:, BASE_FEATURES].copy()

    # Categorical columns: consistent string type, explicit missing level.
    cat_cols = out.select_dtypes(include=["object", "category", "string"]).columns
    for c in cat_cols:
        out[c] = out[c].astype("object").where(out[c].notna(), "missing").astype(str)

    amt = out["transaction_amount"].astype(float)
    bal = out["account_balance"].astype(float)
    emi = out["emi_amount"].astype(float)

    # Ratios: how large is this transaction relative to the customer's means?
    out["amount_to_balance"] = amt / (bal.abs() + 1.0)
    out["amount_to_emi"] = amt / (emi.abs() + 1.0)
    out["emi_to_balance"] = emi / (bal.abs() + 1.0)
    out["amount_x_credit_score"] = amt * out["credit_score"].astype(float)
    out["amount_exceeds_balance"] = (amt > bal).astype("int8")

    # Time: hour/month/weekday are cyclical, and night hours are riskier.
    hour = out["transaction_hour"].astype(float)
    out["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    out["is_night"] = ((hour < 6) | (hour >= 22)).astype("int8")
    month = out["transaction_month"].astype(float)
    out["month_sin"] = np.sin(2 * np.pi * month / 12)
    out["month_cos"] = np.cos(2 * np.pi * month / 12)

    # Categorical interactions (CatBoost also builds combinations internally,
    # but explicit ones help on small trees / early iterations).
    out["channel_x_txn_type"] = out["channel"] + "|" + out["transaction_type"]
    out["merchant_x_channel"] = out["merchant_category"] + "|" + out["channel"]
    out["kyc_x_channel"] = out["kyc_status"] + "|" + out["channel"]
    return out


X = build_features(df)
y = df[TARGET].astype("int8").copy()
categorical_cols = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
print(f"Feature matrix: {X.shape}; categorical: {len(categorical_cols)}")
print(f"Target distribution: {y.value_counts().sort_index().to_dict()}  (fraud rate {y.mean():.4%})")


## 3. Hold out a test set

The test set is touched **once**, at the very end. All tuning, early stopping and threshold selection use the training split only (via cross-validation).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y,
)
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
print(f"Train rows: {len(X_train):,} (fraud {int(y_train.sum()):,}) | Test rows: {len(X_test):,} (fraud {int(y_test.sum()):,})")

neg, pos = int((y_train == 0).sum()), int((y_train == 1).sum())
if pos == 0:
    raise ValueError("Training split contains no fraud examples.")
FULL_IMBALANCE_RATIO = neg / pos
print(f"Imbalance ratio (neg/pos): {FULL_IMBALANCE_RATIO:.1f}")


## 4. Hyperparameter search (Optuna)

* Each trial is scored with **3-fold stratified CV average precision**, with **early stopping** so the number of trees is found automatically.
* The strength of the class weighting is tuned too: `scale_pos_weight = (neg/pos) ** p`, where `p` in [0, 1]. Full weighting is often *not* optimal for PR-AUC.
* Bad trials are pruned after the first fold to save time.

In [ ]:
# Stratified subsample keeps tuning fast on very large data.
if len(X_train) > SEARCH_ROWS:
    X_search, _, y_search, _ = train_test_split(
        X_train, y_train, train_size=SEARCH_ROWS, random_state=RANDOM_STATE, stratify=y_train,
    )
    X_search, y_search = X_search.reset_index(drop=True), y_search.reset_index(drop=True)
else:
    X_search, y_search = X_train, y_train
print(f"Tuning on {len(X_search):,} rows")

tune_cv = StratifiedKFold(n_splits=TUNE_FOLDS, shuffle=True, random_state=RANDOM_STATE)


def make_model(params, iterations, early_stopping=False, thread_count=-1, seed=RANDOM_STATE):
    return CatBoostClassifier(
        **params,
        iterations=iterations,
        loss_function="Logloss",
        eval_metric="PRAUC",
        early_stopping_rounds=100 if early_stopping else None,
        random_seed=seed,
        thread_count=thread_count,
        allow_writing_files=False,
        verbose=False,
    )


def objective(trial: optuna.Trial) -> float:
    power = trial.suggest_float("pos_weight_power", 0.0, 1.0)
    params = {
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.25, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.1, 5.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 5.0),
        "border_count": trial.suggest_categorical("border_count", [64, 128, 254]),
        "one_hot_max_size": trial.suggest_int("one_hot_max_size", 2, 10),
        "scale_pos_weight": float(FULL_IMBALANCE_RATIO ** power),
    }
    scores, best_iters = [], []
    for fold, (tr, va) in enumerate(tune_cv.split(X_search, y_search)):
        model = make_model(params, MAX_ITERATIONS, early_stopping=True)
        model.fit(
            X_search.iloc[tr], y_search.iloc[tr],
            cat_features=categorical_cols,
            eval_set=(X_search.iloc[va], y_search.iloc[va]),
            use_best_model=True,
        )
        proba = model.predict_proba(X_search.iloc[va])[:, 1]
        scores.append(average_precision_score(y_search.iloc[va], proba))
        best_iters.append(model.get_best_iteration())
        trial.report(float(np.mean(scores)), fold)
        if trial.should_prune():
            raise optuna.TrialPruned()
    trial.set_user_attr("mean_best_iteration", int(np.mean(best_iters)))
    return float(np.mean(scores))


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

best_trial = study.best_trial
print(f"Best {TUNE_FOLDS}-fold CV average precision: {best_trial.value:.4f}")
print("Best params:", best_trial.params)
print("Mean best iteration:", best_trial.user_attrs["mean_best_iteration"])


## 5. Final model: 5-fold ensemble on the full training split

* Every training row is used for learning (the old notebook discarded ~36% of the training data).
* A **fixed** number of trees (from tuning, +15% since each model now sees more data) is used, so the **out-of-fold (OOF)** predictions are unbiased and can be used to pick the threshold.
* Test predictions are the **average of the 5 fold models**, which reduces variance.

In [ ]:
best_params = dict(best_trial.params)
power = best_params.pop("pos_weight_power")
best_params["scale_pos_weight"] = float(FULL_IMBALANCE_RATIO ** power)
final_iterations = max(100, int(best_trial.user_attrs["mean_best_iteration"] * 1.15))
print(f"Final iterations per model: {final_iterations}")

final_cv = StratifiedKFold(n_splits=FINAL_FOLDS, shuffle=True, random_state=RANDOM_STATE + 1)
oof_proba = np.zeros(len(X_train))
test_proba = np.zeros(len(X_test))
models = []

for fold, (tr, va) in enumerate(final_cv.split(X_train, y_train), start=1):
    model = make_model(best_params, final_iterations, seed=RANDOM_STATE + fold)
    model.fit(X_train.iloc[tr], y_train.iloc[tr], cat_features=categorical_cols)
    oof_proba[va] = model.predict_proba(X_train.iloc[va])[:, 1]
    test_proba += model.predict_proba(X_test)[:, 1] / FINAL_FOLDS
    models.append(model)
    print(f"Fold {fold}: OOF AP = {average_precision_score(y_train.iloc[va], oof_proba[va]):.4f}")

print(f"\nOverall OOF average precision: {average_precision_score(y_train, oof_proba):.4f}")
print(f"Overall OOF ROC-AUC:           {roc_auc_score(y_train, oof_proba):.4f}")


## 6. Choose the decision threshold (OOF data only)

Two operating points are computed:
* **Max F1** – best balance of precision and recall.
* **High precision** – the lowest threshold that still reaches `MIN_PRECISION_TARGET` precision (fewer false alarms, useful when reviews are expensive).

Pick whichever matches your business cost. If a missed fraud costs far more than a false alarm, lower the threshold instead.

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_train, oof_proba)
f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
thr_f1 = float(thresholds[int(np.argmax(f1))])

ok = np.where(precision[:-1] >= MIN_PRECISION_TARGET)[0]
thr_precision = float(thresholds[ok[0]]) if len(ok) else thr_f1

print(f"Max-F1 threshold:         {thr_f1:.4f}  (OOF F1 = {f1.max():.4f})")
print(f"High-precision threshold: {thr_precision:.4f}  "
      f"({'reached' if len(ok) else 'NOT reachable, fell back to max-F1'} precision >= {MIN_PRECISION_TARGET:.0%})")

# Choose which operating point to deploy:
decision_threshold = thr_f1


## 7. Final evaluation on the untouched test set

In [ ]:
def report(name, threshold):
    pred = (test_proba >= threshold).astype("int8")
    print(f"=== {name} (threshold = {threshold:.4f}) ===")
    print(classification_report(y_test, pred, labels=[0, 1],
                                target_names=["Non-Fraud", "Fraud"], zero_division=0, digits=4))
    print(f"MCC: {matthews_corrcoef(y_test, pred):.4f}\n")
    return pred


test_ap = average_precision_score(y_test, test_proba)
test_auc = roc_auc_score(y_test, test_proba)
baseline = float(y_test.mean())
print(f"Test average precision (PR-AUC): {test_ap:.4f}  | no-skill baseline: {baseline:.4f}  | lift: {test_ap / baseline:.1f}x")
print(f"Test ROC-AUC:                    {test_auc:.4f}\n")

# 95% bootstrap confidence interval for AP, so you know how much to trust the number.
rng = np.random.default_rng(RANDOM_STATE)
y_arr = y_test.to_numpy()
boot = []
for _ in range(300):
    idx = rng.integers(0, len(y_arr), len(y_arr))
    if y_arr[idx].sum() > 0:
        boot.append(average_precision_score(y_arr[idx], test_proba[idx]))
print(f"Test AP 95% CI: [{np.percentile(boot, 2.5):.4f}, {np.percentile(boot, 97.5):.4f}]\n")

test_predictions = report("Selected threshold (max F1)", decision_threshold)
if thr_precision != thr_f1:
    report(f"High-precision (>= {MIN_PRECISION_TARGET:.0%})", thr_precision)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

matrix = confusion_matrix(y_test, test_predictions, labels=[0, 1])
sns.heatmap(matrix, annot=True, fmt=",d", cmap="Blues", ax=axes[0],
            xticklabels=["Non-Fraud", "Fraud"], yticklabels=["Non-Fraud", "Fraud"])
axes[0].set(title="Held-out Test Confusion Matrix", xlabel="Predicted", ylabel="Actual")

p, r, _ = precision_recall_curve(y_test, test_proba)
axes[1].plot(r, p, label=f"CatBoost ensemble (AP={test_ap:.3f})")
axes[1].axhline(baseline, ls="--", color="grey", label=f"No skill ({baseline:.3f})")
axes[1].set(title="Precision-Recall Curve (test)", xlabel="Recall", ylabel="Precision", ylim=(0, 1.02))
axes[1].legend()
fig.tight_layout()
plt.show()


## 8. Feature importance and saving the model

In [ ]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": np.mean([m.get_feature_importance() for m in models], axis=0),
}).sort_values("importance", ascending=False)

plt.figure(figsize=(8, 7))
sns.barplot(data=importance.head(20), x="importance", y="feature", color="steelblue")
plt.title("Top 20 features (mean over ensemble)")
plt.tight_layout()
plt.show()

# Low-importance features can be removed to simplify the model:
print("Features with ~0 importance:", importance.loc[importance["importance"] < 0.1, "feature"].tolist())


In [ ]:
import json

artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)
for i, m in enumerate(models, start=1):
    m.save_model(str(artifact_dir / f"catboost_fold{i}.cbm"))
(artifact_dir / "config.json").write_text(json.dumps({
    "threshold": decision_threshold,
    "feature_columns": X_train.columns.tolist(),
    "categorical_columns": categorical_cols,
    "best_params": best_params,
    "iterations": final_iterations,
}, indent=2))
print(f"Saved {len(models)} models + config.json to {artifact_dir.resolve()}")
print("To score new data: X_new = build_features(new_df); proba = mean(m.predict_proba(X_new)[:, 1] for m in models); flag = proba >= threshold")
